In [ ]:
import pandas as pd

### difference-in-difference

https://www.kaggle.com/code/harrywang/difference-in-differences-in-python/notebook

dif_n_dif = (y_exp_after - y_exp_before) - (y_control_after - y_control_before)  
dif_n_dif -> conf interval -> significance

<u> linear regression: </u>  
y = b0 + b1 * exp_group + b2 * time + b3 * exg_group * time  
exp_group = 1 if exp else 0  
time = 1 if after else 0

y_control_before = b0 ; y_control_after = b0 + b2  
y_exp_before = b0 + b1 ; y_exp_after = b0 + b1 + b2 + b3  
dif_n_dif = (y_exp_after - y_exp_before) - (y_control_after - y_control_before) = b3  

<img src="images/diff_n_diff.png" width="400" align="left">  

In [ ]:
df = pd.read_csv('./data/employment.csv')
df.groupby('state').mean()

In [ ]:
did = (21.096667 - 23.380000) - (20.897249 - 20.430583)
print(did)

In [ ]:
# make total ds
before = df[['state', 'total_emp_feb']]
before['t'] = 0
before.rename(columns={'state' : 'g', 'total_emp_feb' : 'empl_cnt'}, inplace=True)
after = df[['state', 'total_emp_nov']]
after['t'] = 1
after.rename(columns={'state' : 'g', 'total_emp_nov' : 'empl_cnt'}, inplace=True)
ds = pd.concat([before[['g', 't', 'empl_cnt']], after[['g', 't', 'empl_cnt']]], ignore_index=False)
ds['gt'] = ds.g * ds.t

# линейная регрессия
from statsmodels.formula.api import ols
ols = ols('empl_cnt ~ g + t + gt', data=ds).fit()
print(ols.summary())

In [ ]:
# gt = 2.75; p_val = 0.113
# under assumption: parallel trends in con/exp without treatment

In [ ]:
did = []
for _ in range(5000):
    tmp = ds.copy()
    # randomly assign g/t marks (H0 hypothesis)
    tmp['g'] = np.random.permutation(tmp.g.values)
    t1 = tmp[(tmp.t == 0) & (tmp.g == 0)].empl_cnt.mean()
    t2 = tmp[(tmp.t == 0) & (tmp.g == 1)].empl_cnt.mean()
    t3 = tmp[(tmp.t == 1) & (tmp.g == 0)].empl_cnt.mean()
    t4 = tmp[(tmp.t == 1) & (tmp.g == 1)].empl_cnt.mean()
    did_ = (t4 - t3) - (t2 - t1)
    did.append(did_) # h0 hyp, did = 0
# p_value = prob(sample_did extremly than obs_did)
len([j for j in did if abs(j) > abs(obs_did)]) / len(did)

In [ ]:
# p_val совпало с значением в линейной регрессии

In [ ]:
# (!) проверка параллельности трендов в до-тестовый период:
# таким же образом проверяем conf_int_did для t1, t2... < t_exp - должно быть незначительным
# аналогично можно использовать регрессию и показать что фактор gt не играет роли
# https://stats.stackexchange.com/questions/160359/difference-in-difference-method-how-to-test-for-assumption-of-common-trend-betw

### Simple Matching

In [ ]:
# https://www.kaggle.com/code/harrywang/simple-matching-in-python
# https://harrywang.me/psm-did

In [ ]:
# видим, что лечение для курильщика вероятно помогает
# однако впрямую control/test нельзя сравнивать так как разбивка не рандомная
df = pd.read_csv('./data/smoker.csv')
df.groupby(['smoker', 'treatment']).outcome.mean().reset_index()

In [ ]:
# 1:1 match - for each person in treatment, we find a match from the control, 
# i.e., if the person is a smoker, we find a smoker in the control
treatment = df[df.treatment == 1]; control = df[df.treatment == 0]

smokers_cnt = min(treatment[treatment.smoker == 1].shape[0], control[control.smoker == 1].shape[0])
non_smokers_cnt = min(treatment[treatment.smoker == 0].shape[0], control[control.smoker == 0].shape[0])

df1 = treatment[treatment.smoker == 1].sample(smokers_cnt, replace=False)
df2 = treatment[treatment.smoker == 0].sample(non_smokers_cnt, replace=False)
df3 = control[control.smoker == 1].sample(smokers_cnt, replace=False)
df4 = control[control.smoker == 0].sample(non_smokers_cnt, replace=False)
df_matched = pd.concat([df1,df2,df3,df4], ignore_index=True)

df.groupby(['treatment', 'smoker']).outcome.mean()

In [ ]:
# распределение показателей не изменилось
df_matched.groupby(['treatment', 'smoker']).outcome.mean()

In [ ]:
# при этом сравнялся состав участников в группах
df_matched.groupby(['treatment', 'smoker']).count()

In [ ]:
# до нормализации создавался ложный эффект, что лечение повышает долю летальных исходов outcome=1
df.groupby('treatment').outcome.mean()

In [ ]:
# после нормализации видим что все окей
df_matched.groupby('treatment').outcome.mean()

### Propensity Score matching

In [ ]:
# https://harrywang.me/psm-did